# Fine-tune an SAE with C2C box supervision

This interactive notebook configures and launches the existing `sae-training/train_box_sae.py` experiment. The trainer fine-tunes the SAE and a spatial box-prediction head using reconstruction, objectness, and box losses. It uses C2C box targets as working labels, not verified ground truth, and trains both kimono and flower heads together.

The companion `debiasing` workspace must contain prepared C2C targets and captured spatial residuals. Run this on a CUDA GPU node. It starts local Python training; it does not submit a Slurm job.

In [ ]:
from pathlib import Path
import sys
import subprocess
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

NOTEBOOK_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if p.name == "sae-demo-notebooks" and (p / "sae-tuning").exists())
DEBIASING_ROOT = NOTEBOOK_ROOT.parent  # Edit this if debiasing is elsewhere.
SAE_TRAINING = DEBIASING_ROOT / "sae-training"
TRAINER = SAE_TRAINING / "train_box_sae.py"
TARGETS = SAE_TRAINING / "c2c_box_targets.csv"
RESIDUALS = SAE_TRAINING / "residuals"


## Configure training

Choose a fresh run name each time. The existing trainer stops if the output folder already exists, which protects earlier checkpoints.

In [ ]:
run_name = widgets.Text(value="notebook_run", description="Run name:")
epochs = widgets.IntSlider(value=5, min=1, max=20, step=1, description="Epochs:")
sae_lr = widgets.FloatText(value=1e-6, description="SAE LR:")
head_lr = widgets.FloatText(value=1e-3, description="Head LR:")
box_weight = widgets.FloatText(value=5.0, description="Box loss:")
freeze_sae = widgets.Checkbox(value=False, description="Freeze SAE (head-only baseline)")
button = widgets.Button(description="Start fine-tuning", button_style="primary")
output = widgets.Output()

def launch_training(_):
    with output:
        clear_output(wait=True)
        if not TRAINER.exists() or not TARGETS.exists() or not RESIDUALS.exists():
            print("Missing companion files. Prepare targets and capture residuals in the debiasing workspace first.")
            print(f"Expected: {TRAINER}, {TARGETS}, {RESIDUALS}")
            return
        suffix = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        safe_name = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in run_name.value.strip())
        run_dir = SAE_TRAINING / "runs" / f"{safe_name}_{suffix}"
        command = [sys.executable, str(TRAINER), "--targets", str(TARGETS),
                   "--residuals", str(RESIDUALS), "--output", str(run_dir),
                   "--epochs", str(epochs.value), "--sae-lr", str(sae_lr.value),
                   "--head-lr", str(head_lr.value), "--box-weight", str(box_weight.value)]
        if freeze_sae.value:
            command.append("--freeze-sae")
        print("Running:", " ".join(command), flush=True)
        result = subprocess.run(command, cwd=DEBIASING_ROOT, check=False)
        print(f"Exit code: {result.returncode}")
        if result.returncode == 0:
            print(f"Run artifacts: {run_dir}")

button.on_click(launch_training)
display(run_name, epochs, sae_lr, head_lr, box_weight, freeze_sae, button, output)


The training script loads the pretrained SAE checkpoint specified by `backprop/generation_settings.json`. A successful run writes its selected SAE checkpoint, box head, validation history, and held-out predictions under the run directory. Compare against a frozen-SAE run before concluding that fine-tuning improved concept representation.